# Test Notebook for Assigning Reservations to Service Centers

In [ ]:
from geopy.geocoders import Nominatim
from geopy.distance import geodesic

from vehsim.interface import CenterView, ReservationView

# Sample unassigned reservations for testing (center_id is None until assigned)
sample_reservations = [
    ReservationView(id="R000001", created_day=0, street="400 Broad St", city="Seattle", state="WA", zip="98109", center_id=None),
    ReservationView(id="R000002", created_day=2, street="1 Dr Carlton B Goodlett Pl", city="San Francisco", state="CA", zip="94102", center_id=None),
    ReservationView(id="R000003", created_day=5, street="233 S Wacker Dr", city="Chicago", state="IL", zip="60606", center_id=None),
    ReservationView(id="R000004", created_day=6, street="85 Pike St", city="Seattle", state="WA", zip="98101", center_id=None),
]
# Sample delivery centers: two in Chicago (same city as R000003), one in Denver.
# Ids C21+ so they don't clash with the 20 default centers (C01-C20).
sample_centers = [
    CenterView(id="C21", address="1300 S Lake Shore Dr, Chicago, IL 60605", weekly_capacity=40, queue_length=0, opened_day=0),
    CenterView(id="C22", address="5700 S Lake Shore Dr, Chicago, IL 60637", weekly_capacity=25, queue_length=0, opened_day=0),
    CenterView(id="C23", address="1701 Bryant St, Denver, CO 80204", weekly_capacity=30, queue_length=0, opened_day=0),
]

In [4]:
def get_city_state(address):
    address_split = address[:-6].split(', ')
    return (address_split[-2] + ', ' + address_split[-1])

def get_coords(address, geo_agent):
    loc = geo_agent.geocode(address)
        
    if loc:
        return (loc.latitude, loc.longitude)
    else:
        ValueError(f"Delivery center address {address} not found!")

geolocator = Nominatim(user_agent="geo_agent")

center_coords_dict = {}
nearest_center_city = {}

for center in sample_centers:
    center_city_state = get_city_state(center.address)
    center_coords = get_coords(center.address, geolocator)

    if center_city_state in center_coords_dict:
        center_coords_dict[center_city_state][center_coords] = center.id
    else:
        center_coords_dict[center_city_state] = {center_coords: center.id}

for city in center_coords_dict:
    print(f"City: {city}, Number of centers: {len(center_coords_dict[city])}")


City: Chicago, IL, Number of centers: 2
City: Denver, CO, Number of centers: 1


## Convert Reservation Address to LatLon

In [ ]:
def get_closest_center(res_address, center_coords_dict, geo_agent):
    # print(center_coords_dict)
    if len(center_coords_dict) == 1:
        allocated_center = list(center_coords_dict.values())[0]
    
    else:
        closest_center_id = ''
        closest_center_distance = float('inf')

        res_coords = get_coords(res_address, geo_agent)

        for center_coords, center_id in center_coords_dict.items():
            dist = geodesic(res_coords, center_coords).miles
            print(f"Center {center_id} is {dist} from reservation address")
            if dist < closest_center_distance:
                closest_center_id = center_id
                closest_center_distance = dist

            allocated_center = closest_center_id
    return allocated_center

def get_nearest_center_city(res_city_state_coords, center_coords_dict):
    closest_dist = float('inf')
    closest_center_city = ""
    
    for center_city_state, center_coords_ids_dict in center_coords_dict.items():
        # print(f'Checking {center_city_state}')
        for center_coord in center_coords_ids_dict.keys():
            dist = geodesic(res_city_state_coords, center_coord).miles
            # print(f'Distance to a center in {center_city_state} is {dist}')
            if dist < closest_dist:
                closest_dist = dist
                closest_center_city = center_city_state
        # print(f'After checking {center_city_state}, closest city with a delivery center is {closest_center_city}')

    print(f'    Closest city with a delivery center is {closest_center_city}')
    return closest_center_city

reservation_assignments = {}

for res in sample_reservations:
    res_city_state = res.city + ', ' + res.state
    res_address = res.street + ', ' + res_city_state + ' ' + res.zip
    
    print(f"Processing resevation in {res_city_state}")

    if res_city_state in center_coords_dict:
        print("    Reservation is in a city with a delivery center")
        center_city = res_city_state 
    
    elif res_city_state in nearest_center_city:
        print("    Reservation is in a city without a delivery center but in the same city as another reservation")
        center_city = nearest_center_city[res_city_state]
               
    else:
        print("    Reservation is in a new city without a delivery center")
        center_city = get_nearest_center_city(get_coords(res_city_state, geolocator), center_coords_dict)
        nearest_center_city[res_city_state] = center_city            

    reservation_assignments[res.id] = get_closest_center(res_address, center_coords_dict[center_city], geolocator)

print(reservation_assignments)

Processing resevation in Seattle, WA
    Reservation is in a new city without a delivery center
    Closest city with a delivery center is Denver, CO
Processing resevation in San Francisco, CA
    Reservation is in a new city without a delivery center
    Closest city with a delivery center is Denver, CO
Processing resevation in Chicago, IL
    Reservation is in a city with a delivery center
Center C21 is 1.7375215059947524 from reservation address
Center C22 is 6.660957679175542 from reservation address
Processing resevation in Seattle, WA
    Reservation is in a city without a delivery center but in the same city as another reservation
{'R000001': 'C23', 'R000002': 'C23', 'R000003': 'C21', 'R000004': 'C23'}
